# Assignment 4: Probabilistic PCA and Variational Autoencoders

This notebook bridges classical latent variable models and modern deep generative models:
- PCA review and its limitations
- Probabilistic PCA (PPCA): a Gaussian LVM
- The ELBO derivation step-by-step
- The reparameterisation trick
- Variational Autoencoder (VAE) implementation in PyTorch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams['figure.figsize'] = (9, 6)

---
## Part 1: PCA Review

PCA finds the directions of maximum variance in the data by decomposing the data covariance matrix.

Given data $\mathbf{X} \in \mathbb{R}^{N \times D}$ (mean-centred), PCA computes:
- Covariance matrix: $\mathbf{C} = \frac{1}{N} \mathbf{X}^\top \mathbf{X}$
- Eigendecomposition: $\mathbf{C} = \mathbf{V} \mathbf{\Lambda} \mathbf{V}^\top$
- Projection to $M$ dimensions: $\mathbf{z} = \mathbf{V}_M^\top (\mathbf{x} - \boldsymbol{\mu})$

In [ ]:
def pca(X, M):
    """
    Perform PCA on X and return the top-M principal components.

    Args:
        X : np.ndarray (N, D)
        M : int, number of components to keep

    Returns:
        Z       : np.ndarray (N, M), projected data
        W       : np.ndarray (D, M), principal directions (columns = eigenvectors)
        mu      : np.ndarray (D,), data mean
        explained_var : np.ndarray (M,), fraction of variance explained by each component
    """
    # TODO:
    # 1. Compute mean and centre the data
    # 2. Compute empirical covariance C = (1/N) * X_centred.T @ X_centred
    # 3. Eigendecompose C using np.linalg.eigh (returns sorted ascending)
    #    Then reverse to get descending order
    # 4. W = top M eigenvectors
    # 5. Z = X_centred @ W
    # 6. explained_var = top M eigenvalues / sum(all eigenvalues)
    raise NotImplementedError


# Load the digits dataset
digits = load_digits()
X_digits = digits.data.astype(float)  # (1797, 64)
y_digits = digits.target

Z_pca, W_pca, mu_pca, ev_pca = pca(X_digits, M=20)

# Plot cumulative explained variance
plt.plot(np.cumsum(ev_pca) * 100, 'b-o', markersize=4)
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance (%)')
plt.title('PCA: explained variance')
plt.grid(True)
plt.show()

print(f'First 2 PCs explain {(ev_pca[:2].sum()*100):.1f}% of variance')
print(f'First 10 PCs explain {(ev_pca[:10].sum()*100):.1f}% of variance')

In [ ]:
# Visualise the 2D PCA embedding
Z_2d, _, _, _ = pca(X_digits, M=2)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(Z_2d[:, 0], Z_2d[:, 1], c=y_digits, cmap='tab10', s=8, alpha=0.7)
plt.colorbar(scatter, label='Digit')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.title('PCA: 2D projection of digits')
plt.show()

In [ ]:
def pca_reconstruct(Z, W, mu):
    """
    Reconstruct data from PCA latent codes.

    Args:
        Z  : np.ndarray (N, M)
        W  : np.ndarray (D, M)
        mu : np.ndarray (D,)

    Returns:
        X_rec : np.ndarray (N, D)
    """
    # TODO: X_rec = Z @ W.T + mu
    raise NotImplementedError


# Reconstruct from different numbers of components and compare
fig, axes = plt.subplots(3, 6, figsize=(12, 7))
n_components_list = [1, 5, 10, 20, 40, 64]

for row_idx in range(3):
    sample_idx = row_idx * 200
    axes[row_idx, 0].imshow(X_digits[sample_idx].reshape(8, 8), cmap='gray')
    axes[row_idx, 0].set_title('Original')
    axes[row_idx, 0].axis('off')

    for col_idx, M in enumerate(n_components_list[1:], start=1):
        Z_m, W_m, mu_m, _ = pca(X_digits, M)
        X_rec = pca_reconstruct(Z_m, W_m, mu_m)
        axes[row_idx, col_idx].imshow(X_rec[sample_idx].reshape(8, 8), cmap='gray')
        axes[row_idx, col_idx].set_title(f'M={M}')
        axes[row_idx, col_idx].axis('off')

plt.suptitle('PCA reconstruction quality vs number of components')
plt.tight_layout()
plt.show()

---
## Part 2: Probabilistic PCA (PPCA)

PPCA is a **generative model** that explains the data via a latent variable $\mathbf{z} \in \mathbb{R}^M$:

$$p(\mathbf{z}) = \mathcal{N}(\mathbf{z} \mid \mathbf{0}, \mathbf{I})$$
$$p(\mathbf{x} \mid \mathbf{z}) = \mathcal{N}(\mathbf{x} \mid \mathbf{W}\mathbf{z} + \boldsymbol{\mu}, \sigma^2 \mathbf{I})$$

This gives a **marginal** (integrating out $z$):
$$p(\mathbf{x}) = \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}, \mathbf{W}\mathbf{W}^\top + \sigma^2 \mathbf{I})$$

PPCA recovers standard PCA in the limit $\sigma^2 \to 0$.

The MLE for PPCA has a closed form:
$$\mathbf{W}_{\text{MLE}} = \mathbf{V}_M (\mathbf{\Lambda}_M - \sigma^2 \mathbf{I})^{1/2} \mathbf{R}$$
$$\sigma^2_{\text{MLE}} = \frac{1}{D - M} \sum_{j=M+1}^{D} \lambda_j$$

where $\mathbf{V}_M$ are the top-$M$ eigenvectors, $\mathbf{\Lambda}_M$ the corresponding eigenvalues, and $\mathbf{R}$ any orthogonal matrix (typically $\mathbf{I}$).

In [ ]:
def fit_ppca(X, M):
    """
    Fit PPCA by closed-form MLE.

    Args:
        X : np.ndarray (N, D)
        M : int, latent dimensionality

    Returns:
        W      : np.ndarray (D, M)
        mu     : np.ndarray (D,)
        sigma2 : float
        lambdas: np.ndarray (D,), all eigenvalues (descending)
    """
    N, D = X.shape
    # TODO:
    # 1. Compute mean mu = X.mean(axis=0)
    # 2. Centre and compute covariance C
    # 3. Eigendecompose C (use np.linalg.eigh, reverse for descending order)
    # 4. sigma2_mle = mean of the D-M smallest eigenvalues
    # 5. W_mle = V_M @ diag(sqrt(lambda_M - sigma2)) (set R = I)
    #    Clamp (lambda_M - sigma2) >= 0 before sqrt to avoid numerical issues
    raise NotImplementedError


def ppca_log_likelihood(X, W, mu, sigma2):
    """
    Compute the PPCA marginal log-likelihood:
        ln p(X) = sum_n ln N(x_n | mu, C)  where C = W @ W.T + sigma2 * I

    Args:
        X      : np.ndarray (N, D)
        W      : np.ndarray (D, M)
        mu     : np.ndarray (D,)
        sigma2 : float

    Returns:
        float
    """
    D = W.shape[0]
    # TODO:
    # 1. Compute C = W @ W.T + sigma2 * np.eye(D)
    # 2. Use multivariate Gaussian log-likelihood:
    #    log p(x_n) = -D/2*log(2pi) - 0.5*log|C| - 0.5*(x-mu)^T C^{-1} (x-mu)
    # 3. Sum over n
    # Hint: use np.linalg.slogdet for the log-determinant,
    #       np.linalg.solve(C, (X-mu).T) for the quadratic term
    raise NotImplementedError


# Test on digits
M = 10
W_ppca, mu_ppca, sigma2_ppca, lambdas_ppca = fit_ppca(X_digits, M)
ll_ppca = ppca_log_likelihood(X_digits, W_ppca, mu_ppca, sigma2_ppca)

print(f'PPCA with M={M}: log-likelihood = {ll_ppca:.2f}')
print(f'Estimated noise variance sigma^2 = {sigma2_ppca:.4f}')

In [ ]:
def ppca_encode(X, W, mu, sigma2):
    """
    Compute the posterior mean E[z|x] under PPCA.

    The posterior is:
        p(z|x) = N(z | M^{-1} W^T (x - mu), sigma^2 M^{-1})
    where M = W^T W + sigma^2 I_M

    Return only the posterior mean (the 'encoding').

    Args:
        X      : np.ndarray (N, D)
        W      : np.ndarray (D, M)
        mu     : np.ndarray (D,)
        sigma2 : float

    Returns:
        Z : np.ndarray (N, M)
    """
    M_mat = W.T @ W + sigma2 * np.eye(W.shape[1])
    # TODO: return np.linalg.solve(M_mat, W.T @ (X - mu).T).T
    raise NotImplementedError


# Compare PPCA and PCA 2D embeddings
W_ppca2, mu_ppca2, sigma2_ppca2, _ = fit_ppca(X_digits, M=2)
Z_ppca2 = ppca_encode(X_digits, W_ppca2, mu_ppca2, sigma2_ppca2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, Z, title in [(axes[0], Z_2d, 'PCA'), (axes[1], Z_ppca2, 'PPCA')]:
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=y_digits, cmap='tab10', s=8, alpha=0.7)
    plt.colorbar(sc, ax=ax, label='Digit')
    ax.set_title(f'{title}: 2D latent space')
    ax.set_xlabel('z1')
    ax.set_ylabel('z2')
plt.tight_layout()
plt.show()

### 2.1 Why PPCA matters

PCA is a **deterministic** projection. PPCA adds a **probabilistic** noise model:
- It gives us a proper likelihood, so we can compare models
- It handles missing data naturally
- It is the linear special case of the **VAE** we build in Part 3

**Question**: If $\sigma^2 \to 0$, what does PPCA reduce to? What happens if $\sigma^2 \to \infty$?

**Your answer**: TODO

---
## Part 3: The ELBO Derivation

For any latent variable model, the log-likelihood is intractable when the integral over $z$ has no closed form:

$$\ln p(\mathbf{x} | \mathbf{w}) = \ln \int p(\mathbf{x} | \mathbf{z}, \mathbf{w}) \, p(\mathbf{z}) \, d\mathbf{z}$$

We introduce a variational distribution $q(\mathbf{z} | \mathbf{x}, \boldsymbol{\phi})$ and derive the ELBO:

$$\ln p(\mathbf{x} | \mathbf{w}) = \underbrace{\mathbb{E}_{q}[\ln p(\mathbf{x} | \mathbf{z}, \mathbf{w})] - \text{KL}[q(\mathbf{z} | \mathbf{x}, \boldsymbol{\phi}) \| p(\mathbf{z})]}_{{\mathcal{L}(\mathbf{w}, \boldsymbol{\phi}; \mathbf{x}) \leq \ln p(\mathbf{x}|\mathbf{w})}} + \underbrace{\text{KL}[q(\mathbf{z} | \mathbf{x}, \boldsymbol{\phi}) \| p(\mathbf{z} | \mathbf{x}, \mathbf{w})]}_{{\geq 0}}$$

The ELBO $\mathcal{L}$ has two terms:
1. **Reconstruction term**: $\mathbb{E}_q[\ln p(\mathbf{x} | \mathbf{z}, \mathbf{w})]$ — how well can we reconstruct $x$ from $z$?
2. **KL term**: $-\text{KL}[q(\mathbf{z} | \mathbf{x}) \| p(\mathbf{z})]$ — how close is the approximate posterior to the prior?

In [ ]:
# Analytical KL between two Gaussians — needed for the VAE KL term
def kl_gaussian_diagonal(mu_q, log_var_q):
    """
    KL divergence from N(mu_q, diag(exp(log_var_q))) to N(0, I).

    Closed form:
        KL = 0.5 * sum( exp(log_var_q) + mu_q^2 - 1 - log_var_q )

    This is KL[q || p] where p = N(0, I).

    Args:
        mu_q      : np.ndarray (M,) or (N, M)
        log_var_q : np.ndarray same shape as mu_q

    Returns:
        float (or ndarray if batched)
    """
    # TODO: implement the closed-form KL formula
    raise NotImplementedError


# Sanity check: KL[N(0,I) || N(0,I)] should be 0
mu_test = np.zeros(5)
log_var_test = np.zeros(5)
print(f'KL[N(0,I) || N(0,I)] = {kl_gaussian_diagonal(mu_test, log_var_test):.4f}  (expected 0)')

# KL should increase as q moves away from prior
mu_test2 = np.array([2.0, 0.0, 0.0, 0.0, 0.0])
print(f'KL[N([2,0,...],I) || N(0,I)] = {kl_gaussian_diagonal(mu_test2, log_var_test):.4f}  (expected 2.0)')

---
## Part 4: Reparameterisation Trick

To backpropagate through a stochastic node $\mathbf{z} \sim q(\mathbf{z} | \mathbf{x})$, we rewrite the sample as a **deterministic function** of the input and noise:

$$\mathbf{z} = \boldsymbol{\mu}_\phi(\mathbf{x}) + \boldsymbol{\sigma}_\phi(\mathbf{x}) \odot \boldsymbol{\varepsilon}, \quad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

Gradients flow through $\boldsymbol{\mu}_\phi$ and $\boldsymbol{\sigma}_\phi$ (encoder parameters), not through the sampling operation.

In [ ]:
def reparameterise(mu, log_var):
    """
    Sample z ~ N(mu, diag(exp(log_var))) using the reparameterisation trick.

    Args:
        mu      : np.ndarray (N, M)
        log_var : np.ndarray (N, M)

    Returns:
        z : np.ndarray (N, M)
    """
    # TODO:
    # 1. Sample epsilon ~ N(0, I) with same shape as mu
    # 2. z = mu + sqrt(exp(log_var)) * epsilon
    #    Note: sqrt(exp(log_var)) = exp(0.5 * log_var)
    raise NotImplementedError


# Verify: samples should have correct distribution
mu_test    = np.array([[1.0, -2.0]])
log_var_test = np.array([[0.0, np.log(4.0)]])  # std = [1, 2]

z_samples = np.vstack([reparameterise(mu_test, log_var_test) for _ in range(2000)])
print(f'Sample mean: {z_samples.mean(axis=0)}  (expected [1, -2])')
print(f'Sample std:  {z_samples.std(axis=0)}   (expected [1,  2])')

---
## Part 5: Variational Autoencoder (VAE)

The VAE combines three ideas from Lecture 5:
1. **ELBO**: tractable lower bound to the log-likelihood
2. **Amortised variational inference**: a single encoder network $q_{\phi}(\mathbf{z}|\mathbf{x})$ replaces per-datapoint optimisation
3. **Reparameterisation trick**: enables backpropagation through the stochastic latent variable

Model:
- Prior: $p(\mathbf{z}) = \mathcal{N}(\mathbf{0}, \mathbf{I})$
- Decoder (generative): $p_\mathbf{w}(\mathbf{x} | \mathbf{z})$ — neural network mapping $z \to x$
- Encoder (amortised posterior): $q_\phi(\mathbf{z} | \mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}_\phi(\mathbf{x}), \text{diag}(\boldsymbol{\sigma}^2_\phi(\mathbf{x})))$

ELBO per datapoint:
$$\mathcal{L}(\mathbf{w}, \boldsymbol{\phi}; \mathbf{x}) = \mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})}[\ln p_\mathbf{w}(\mathbf{x} | \mathbf{z})] - \text{KL}[q_\phi(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z})]$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
class Encoder(nn.Module):
    """
    Encoder network: maps input x to (mu_z, log_var_z).

    Architecture:
        Linear(input_dim, hidden_dim) -> ReLU
        Linear(hidden_dim, hidden_dim) -> ReLU
        Two parallel heads:
            Linear(hidden_dim, latent_dim)  -> mu_z
            Linear(hidden_dim, latent_dim)  -> log_var_z
    """

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        # TODO: define the layers described above
        # self.shared = nn.Sequential(...)
        # self.fc_mu  = nn.Linear(...)
        # self.fc_log_var = nn.Linear(...)
        raise NotImplementedError

    def forward(self, x):
        """
        Args:
            x : Tensor (batch, input_dim)

        Returns:
            mu      : Tensor (batch, latent_dim)
            log_var : Tensor (batch, latent_dim)
        """
        # TODO: pass x through shared layers, then the two heads
        raise NotImplementedError


class Decoder(nn.Module):
    """
    Decoder network: maps latent code z to reconstruction x_hat.

    Architecture:
        Linear(latent_dim, hidden_dim) -> ReLU
        Linear(hidden_dim, hidden_dim) -> ReLU
        Linear(hidden_dim, input_dim)  -> Sigmoid  (for [0,1] pixel values)
    """

    def __init__(self, latent_dim, hidden_dim, output_dim):
        super().__init__()
        # TODO: define the decoder layers
        raise NotImplementedError

    def forward(self, z):
        """
        Args:
            z : Tensor (batch, latent_dim)

        Returns:
            x_hat : Tensor (batch, output_dim), values in [0,1]
        """
        # TODO: pass z through decoder layers
        raise NotImplementedError


class VAE(nn.Module):
    """
    Full VAE: encoder + reparameterisation + decoder.
    """

    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)

    def reparameterise(self, mu, log_var):
        """
        Reparameterisation trick.

        Args:
            mu      : Tensor (batch, latent_dim)
            log_var : Tensor (batch, latent_dim)

        Returns:
            z : Tensor (batch, latent_dim)
        """
        # TODO:
        # During training: z = mu + std * epsilon, epsilon ~ N(0,I)
        # std = exp(0.5 * log_var)
        # Use torch.randn_like(mu) for epsilon
        raise NotImplementedError

    def forward(self, x):
        """
        Args:
            x : Tensor (batch, input_dim)

        Returns:
            x_hat   : Tensor (batch, input_dim), reconstructed x
            mu      : Tensor (batch, latent_dim)
            log_var : Tensor (batch, latent_dim)
        """
        # TODO:
        # 1. Encode: mu, log_var = self.encoder(x)
        # 2. Sample: z = self.reparameterise(mu, log_var)
        # 3. Decode: x_hat = self.decoder(z)
        raise NotImplementedError

In [ ]:
def elbo_loss(x, x_hat, mu, log_var):
    """
    Compute the negative ELBO (loss to minimise):

        loss = reconstruction_loss + kl_loss

    where:
        reconstruction_loss = sum of binary cross-entropy per pixel, averaged over batch
            F.binary_cross_entropy(x_hat, x, reduction='sum') / batch_size
        kl_loss = KL[q(z|x) || N(0,I)], averaged over batch
            = 0.5 * sum(exp(log_var) + mu^2 - 1 - log_var) / batch_size

    Args:
        x       : Tensor (batch, D), input (in [0,1])
        x_hat   : Tensor (batch, D), reconstruction
        mu      : Tensor (batch, M)
        log_var : Tensor (batch, M)

    Returns:
        loss   : scalar Tensor
        recon  : scalar Tensor, reconstruction term
        kl     : scalar Tensor, KL term
    """
    batch_size = x.shape[0]
    # TODO: implement reconstruction loss (binary cross-entropy, sum then divide by batch)
    recon = None  # TODO

    # TODO: implement KL loss
    # kl = 0.5 * torch.sum(torch.exp(log_var) + mu**2 - 1 - log_var) / batch_size
    kl = None  # TODO

    loss = recon + kl
    return loss, recon, kl

In [ ]:
# Prepare digits data for VAE
X_norm = X_digits / 16.0  # scale to [0, 1]
X_tensor = torch.FloatTensor(X_norm).to(device)
dataset = TensorDataset(X_tensor)
loader  = DataLoader(dataset, batch_size=64, shuffle=True)

# Hyperparameters
INPUT_DIM  = 64   # 8x8 digits
HIDDEN_DIM = 256
LATENT_DIM = 8
LR         = 1e-3
N_EPOCHS   = 60

vae = VAE(INPUT_DIM, HIDDEN_DIM, LATENT_DIM).to(device)
optimiser = torch.optim.Adam(vae.parameters(), lr=LR)

train_losses, recon_losses, kl_losses = [], [], []

for epoch in range(N_EPOCHS):
    epoch_loss = epoch_recon = epoch_kl = 0.0
    for (x_batch,) in loader:
        optimiser.zero_grad()

        # TODO:
        # 1. Forward pass: x_hat, mu, log_var = vae(x_batch)
        # 2. Compute loss: loss, recon, kl = elbo_loss(x_batch, x_hat, mu, log_var)
        # 3. Backward pass and optimiser step
        raise NotImplementedError

        epoch_loss  += loss.item()
        epoch_recon += recon.item()
        epoch_kl    += kl.item()

    n_batches = len(loader)
    train_losses.append(epoch_loss  / n_batches)
    recon_losses.append(epoch_recon / n_batches)
    kl_losses.append(epoch_kl    / n_batches)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{N_EPOCHS} | '
              f'Loss={train_losses[-1]:.2f} '
              f'Recon={recon_losses[-1]:.2f} '
              f'KL={kl_losses[-1]:.2f}')

In [ ]:
# Plot training curves
epochs = range(1, N_EPOCHS + 1)
plt.figure(figsize=(10, 4))
plt.plot(epochs, train_losses, label='Total ELBO loss')
plt.plot(epochs, recon_losses, label='Reconstruction')
plt.plot(epochs, kl_losses,   label='KL divergence')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE Training')
plt.legend()
plt.grid(True)
plt.show()

### 5.1 Reconstructions

In [ ]:
vae.eval()
with torch.no_grad():
    x_sample = X_tensor[:10]
    x_hat, mu, log_var = vae(x_sample)

fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(10):
    axes[0, i].imshow(x_sample[i].cpu().numpy().reshape(8, 8), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(x_hat[i].cpu().numpy().reshape(8, 8), cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=10)
axes[1, 0].set_ylabel('Recon', fontsize=10)
plt.suptitle('VAE Reconstructions')
plt.tight_layout()
plt.show()

### 5.2 Generating new digits

In [ ]:
def generate_samples_vae(vae, n_samples, device):
    """
    Generate new samples from the VAE by sampling from the prior p(z) = N(0, I)
    and passing through the decoder.

    Args:
        vae       : trained VAE model
        n_samples : int
        device    : str

    Returns:
        samples : np.ndarray (n_samples, input_dim)
    """
    vae.eval()
    with torch.no_grad():
        # TODO:
        # 1. Sample z ~ N(0, I): torch.randn(n_samples, LATENT_DIM)
        # 2. Pass through decoder: x_gen = vae.decoder(z)
        # 3. Convert to numpy
        raise NotImplementedError


generated = generate_samples_vae(vae, 20, device)

fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(20):
    ax = axes[i // 10, i % 10]
    ax.imshow(generated[i].reshape(8, 8), cmap='gray')
    ax.axis('off')
plt.suptitle('VAE Generated Digits (samples from p(z) = N(0,I))')
plt.tight_layout()
plt.show()

### 5.3 Latent space visualisation

In [ ]:
vae.eval()
with torch.no_grad():
    mu_all, log_var_all = vae.encoder(X_tensor)
    mu_all = mu_all.cpu().numpy()

# Project to 2D using PCA on the latent means for visualisation
Z_latent_2d, _, _, _ = pca(mu_all, M=2)

plt.figure(figsize=(9, 7))
sc = plt.scatter(Z_latent_2d[:, 0], Z_latent_2d[:, 1], c=y_digits, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(sc, label='Digit')
plt.xlabel('Latent PC 1')
plt.ylabel('Latent PC 2')
plt.title('VAE latent space (posterior means, projected to 2D)')
plt.show()

### 5.4 Interpolation in latent space

A good latent space should support **smooth interpolation**: moving along a straight line in $z$-space should correspond to a smooth transition in $x$-space.

In [ ]:
def interpolate_latent(vae, x1, x2, n_steps=8, device='cpu'):
    """
    Interpolate between two data points x1 and x2 in latent space.

    Steps:
        1. Encode x1 and x2 to get their posterior means z1 and z2
        2. Linearly interpolate: z_t = (1-t)*z1 + t*z2 for t in [0,1]
        3. Decode each z_t

    Args:
        vae     : trained VAE
        x1, x2  : np.ndarray (D,), data points
        n_steps : int

    Returns:
        images : np.ndarray (n_steps, D)
    """
    vae.eval()
    with torch.no_grad():
        # TODO: implement the interpolation
        raise NotImplementedError


# Find a '1' and a '7' in the dataset
idx_1 = np.where(y_digits == 1)[0][0]
idx_7 = np.where(y_digits == 7)[0][0]

interp_images = interpolate_latent(
    vae, X_norm[idx_1], X_norm[idx_7], n_steps=10, device=device
)

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(interp_images[i].reshape(8, 8), cmap='gray')
    ax.axis('off')
plt.suptitle('Latent space interpolation: 1 → 7')
plt.tight_layout()
plt.show()

---
## Part 6: Comparing PPCA and VAE

Reflect on the connections and differences between PPCA and VAE.

| Aspect | PPCA | VAE |
|--------|------|-----|
| Prior $p(z)$ | $\mathcal{N}(0, I)$ | $\mathcal{N}(0, I)$ |
| Decoder $p(x|z)$ | Linear: $\mathcal{N}(Wz+\mu, \sigma^2 I)$ | TODO |
| Encoder $q(z|x)$ | Closed form (linear) | TODO |
| Training | Closed-form MLE | TODO |
| Generative ability | TODO | TODO |
| Scalability | TODO | TODO |

---
## Part 7 (Bonus): KL Annealing

A common training trick for VAEs is **KL annealing**: start with a small weight on the KL term and gradually increase it. This helps the model learn a good reconstruction before the KL term forces the posterior to match the prior.

In [ ]:
def elbo_loss_annealed(x, x_hat, mu, log_var, beta):
    """
    Beta-VAE loss: reconstruction + beta * KL.

    Beta < 1 downweights the KL (annealing); beta > 1 encourages disentanglement.

    Args:
        x, x_hat, mu, log_var : same as elbo_loss
        beta : float, weight on KL term

    Returns:
        loss, recon, kl
    """
    # TODO: modify elbo_loss to include the beta weighting
    raise NotImplementedError


# TODO (exploration):
# Train a VAE with KL annealing:
#   - beta starts at 0 and linearly increases to 1 over the first 30 epochs
#   - Compare the final reconstructions and generated samples to the standard VAE
#   - Does annealing help? Look at both the loss curves and the quality of generations.

raise NotImplementedError

---
## Summary

In this notebook you:
- Implemented PCA from scratch and visualised reconstruction quality vs component count
- Derived and fitted PPCA as a principled probabilistic extension of PCA
- Derived the closed-form KL divergence between two diagonal Gaussians
- Implemented the reparameterisation trick
- Built and trained a full VAE (encoder + reparameterisation + decoder + ELBO loss)
- Visualised the latent space and generated new digit samples
- Explored latent space interpolation

**The full generative modelling pipeline**:

```
Data → Encoder → (mu, log_var) → Reparameterise → z → Decoder → Reconstruction
                                                  ↑
                                        Prior p(z) = N(0,I)
                              KL pushes posterior toward prior
```